# Music Annotation Project V8.5 — Notebook 43 REBUILT
## Controlled isotropic ownership-context experiment

**Runtime: GPU. Development-only.**

This notebook is a clean, self-contained rebuild. It does not depend on hidden globals from Notebook 41.

### Frozen paired comparison

- **CONTROL:** 33×33 + 97×97 observable context
- **TREATMENT:** the identical 33×33 + 97×97 representation **plus** a 193×193 isotropic outer-context branch, downsampled to 97×97
- **Only intended experimental difference:** addition of the 193×193 branch

The candidate population, labels, source-group folds, losses, training schedule, cross-fitted calibration, selection rule, concentration controls and gates are shared. Clean truth is used only after the candidate manifest is locked. V7.8 remains unchanged except where a selected group copies exact observed bytes.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow scikit-learn torch torchvision matplotlib joblib


## 1. Configuration, pre-registration and hard design assertions


In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONHASHSEED']='43001'
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, random, shutil, sys, platform, gc as py_gc, zipfile, time
import cv2, torch, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V82_ROOT=ROOT/'observed_print_classifier_v8_2'
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
OUTROOT=ROOT/'observed_print_classifier_v8_5_notebook43_rebuilt'
SOURCE_V82_RUN_OVERRIDE=None
SEED=43001
LOWER_P=.50; UPPER_P=.75; MIN_GROUP_PIXELS=2
LOCAL=33; MEDIUM=97; OUTER=193; OUTER_MODEL_SIZE=97
BATCH=96; EPOCHS=14; LR=8e-4; WD=2e-4; BOOT=10000; VALUE_RADIUS=12
SAFETY_T=.995; VALUE_T=.90; MAX_PAGE_FRACTION=.10; MAX_COMPONENT_FRACTION=.20
CHANNELS=['obs','norm','mask','depth','horizontal','vertical','compact','preserve','remove','ambiguous','edge','p_print']
ARMS=['control_33_97','treatment_33_97_193']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE!='cuda': raise RuntimeError('Notebook 43 requires a GPU runtime.')
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False

assert LOCAL==33 and MEDIUM==97 and OUTER==193 and OUTER_MODEL_SIZE==97
assert ARMS==['control_33_97','treatment_33_97_193']
assert len(CHANNELS)==12 and 'p_print' in CHANNELS

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_nb43_REBUILT_33_97_vs_33_97_193'
OUT=OUTROOT/RUN; INPUTS=OUT/'inputs'; TABLES=OUT/'tables'; MODELS=OUT/'models'; CALIBRATORS=OUT/'calibrators'; WHOLE=OUT/'whole_pages'; AUDIT=OUT/'audit'; VISUAL=OUT/'visual_audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,TABLES,MODELS,CALIBRATORS,WHOLE,AUDIT,VISUAL,UPLOAD]: p.mkdir(parents=True,exist_ok=False)

PRE={
 'version':'8.5-notebook43-rebuilt-controlled-isotropic-context',
 'development_only':True,'promotion_authority':False,
 'scientific_question':'Does adding 193x193 isotropic outer context to the existing 33x33+97x97 representation improve source-group OOF safety/value discrimination?',
 'control':'33x33 + 97x97 observable context',
 'treatment':'33x33 + 97x97 observable context + 193x193 isotropic outer context downsampled to 97x97',
 'single_experimental_variable':'addition of the 193x193 isotropic outer-context branch',
 'arms':ARMS,'patches':{'local':LOCAL,'medium':MEDIUM,'outer_source':OUTER,'outer_model_input':OUTER_MODEL_SIZE},
 'channels':CHANNELS,'probability_interval':[LOWER_P,UPPER_P],
 'grouping':{'connectivity':8,'minimum_pixels':MIN_GROUP_PIXELS},
 'safety_threshold':SAFETY_T,'value_threshold':VALUE_T,
 'semantic_labels_as_inputs':False,'seed':SEED
}
(OUT/'01_pre_registration.json').write_text(json.dumps(PRE,indent=2))
print('CONTROL   : 33x33 + 97x97')
print('TREATMENT : 33x33 + 97x97 + 193x193')
print('VARIABLE  : addition of 193x193 isotropic context only')
print('DESIGN CHECK PASSED')
print('Output:',OUT)


## 2. Resolve one complete frozen V8.2 development run


In [ ]:
def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def oh(x): return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def ah(a):
 a=np.ascontiguousarray(a);return hashlib.sha256(str(a.dtype).encode()+str(a.shape).encode()+a.tobytes()).hexdigest()
def latest(root,required):
 hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in required)] if Path(root).exists() else []
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

SOURCE=Path(SOURCE_V82_RUN_OVERRIDE) if SOURCE_V82_RUN_OVERRIDE else latest(V82_ROOT,['experiment_identity.json','tables/all_eligible_oof_predictions.parquet','whole_pages'])
if SOURCE is None: raise RuntimeError('No complete V8.2 development run found.')
ident=json.loads((SOURCE/'experiment_identity.json').read_text())
if ident.get('version')!='8.2-development-whole-page' or ident.get('independent_data_used') is not False:
 raise RuntimeError('Resolved source is not the required development-only V8.2 run.')
V75=Path(ident['v7_5_reference'])
pred_path=SOURCE/'tables/all_eligible_oof_predictions.parquet'
full=pd.read_parquet(pred_path).reset_index(drop=True)
required={'page','source_group','y','x','truth','fold','raw_print','p_print'}
missing=required-set(full.columns)
if missing: raise RuntimeError(f'V8.2 prediction table missing {sorted(missing)}')
for c in ['y','x','truth','fold']: full[c]=pd.to_numeric(full[c],errors='raise').astype(int)
for c in ['raw_print','p_print']: full[c]=pd.to_numeric(full[c],errors='raise').astype(float)
if full.duplicated(['page','y','x']).any(): raise RuntimeError('Duplicate page/y/x predictions.')
if not np.isfinite(full[['raw_print','p_print']].to_numpy()).all(): raise RuntimeError('Non-finite V8.2 predictions.')
record={'source':str(SOURCE),'source_identity_sha256':sha(SOURCE/'experiment_identity.json'),'predictions_sha256':sha(pred_path),'v75':str(V75)}
(OUT/'source_record.json').write_text(json.dumps(record,indent=2))
print(json.dumps(record,indent=2))


## 3. Build authoritative observable page context and lock candidate groups


In [ ]:
status=pd.read_csv(V75/'tables/cache_status.csv')
cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
ctx={}; rows=[]; pix=[]
required_arrays=['mask','geometry_t80','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']

for page,g in full.groupby('page',sort=True):
 if page not in cache: raise RuntimeError(f'{page}: V7.5 cache mapping missing')
 with np.load(cache[page]) as z:
  miss=[k for k in required_arrays if k not in z.files]
  if miss: raise RuntimeError(f'{page}: cache missing {miss}')
  a={k:z[k].copy() for k in required_arrays}
 obs=cv2.imread(str(SOURCE/'whole_pages'/page/'observed.png'),0)
 base=cv2.imread(str(SOURCE/'whole_pages'/page/'v7_8.png'),0)
 mask=a['mask']>0
 if obs is None or base is None or obs.shape!=base.shape or any(v.shape!=obs.shape for v in a.values()):
  raise RuntimeError(f'{page}: unreadable or shape-inconsistent page context')
 if not np.array_equal(base[~mask],obs[~mask]): raise RuntimeError(f'{page}: V7.8 outside-mask mismatch')
 visible=(~mask)&(obs>=200)
 num=cv2.GaussianBlur(obs.astype(np.float32)*visible,(0,0),45.0)
 den=cv2.GaussianBlur(visible.astype(np.float32),(0,0),45.0)
 fallback=cv2.GaussianBlur(obs,(0,0),45.0)
 paper=np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)
 norm=cv2.divide(obs,np.maximum(paper,1),scale=255)
 depth=cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5).astype(np.float32)
 edge=cv2.Canny(obs,70,170)>0
 prob=np.zeros(obs.shape,np.float32); eligible=np.zeros(obs.shape,bool)
 ys=g.y.to_numpy(int); xs=g.x.to_numpy(int)
 prob[ys,xs]=g.p_print.to_numpy(np.float32); eligible[ys,xs]=True
 band=eligible&(prob>=LOWER_P)&(prob<UPPER_P)
 n,lab,stats,_=cv2.connectedComponentsWithStats(band.astype(np.uint8),8)
 _,mcomp,_,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8)
 d={'obs':obs,'baseline':base,'mask':mask,'norm':norm,'depth':depth,'edge':edge,'p_print':prob,
    'horizontal':a['g80_horizontal']>0,'vertical':a['g80_vertical']>0,'compact':a['g80_compact']>0,
    'preserve':a['g80_preserve']>0,'remove':a['g80_remove']>0,'ambiguous':a['g80_ambiguous']>0,
    'candidate_labels':lab,'mask_component':mcomp}
 missing_channels=[k for k in CHANNELS if k not in d]
 if missing_channels: raise RuntimeError(f'{page}: observable context missing {missing_channels}')
 ctx[page]=d
 for cid in range(1,n):
  area=int(stats[cid,cv2.CC_STAT_AREA])
  if area<MIN_GROUP_PIXELS: continue
  yy,xx=np.where(lab==cid); pr=prob[yy,xx]; mc=mcomp[yy,xx]
  primary=int(pd.Series(mc[mc>0]).mode().iloc[0]) if np.any(mc>0) else 0
  gid=f'{page}:g{cid:05d}'
  rows.append({'group_id':gid,'page':page,'source_group':str(g.source_group.iloc[0]),'fold':int(g.fold.iloc[0]),
   'candidate_group_id':cid,'mask_component_id':primary,'pixels':area,'centre_y':float(yy.mean()),'centre_x':float(xx.mean()),
   'p_mean':float(pr.mean()),'p_min':float(pr.min()),'p_max':float(pr.max()),'p_std':float(pr.std()),
   'mask_depth_mean':float(depth[yy,xx].mean()),'edge_rate':float(edge[yy,xx].mean()),
   'horizontal_rate':float(d['horizontal'][yy,xx].mean()),'vertical_rate':float(d['vertical'][yy,xx].mean()),
   'compact_rate':float(d['compact'][yy,xx].mean()),'ambiguous_rate':float(d['ambiguous'][yy,xx].mean())})
  pix.extend({'group_id':gid,'page':page,'y':int(y),'x':int(x)} for y,x in zip(yy,xx))

cand=pd.DataFrame(rows).sort_values(['source_group','page','candidate_group_id']).reset_index(drop=True)
pixels=pd.DataFrame(pix).sort_values(['group_id','y','x']).reset_index(drop=True)
if cand.empty: raise RuntimeError('No registered candidate groups.')
if cand.group_id.duplicated().any(): raise RuntimeError('Duplicate candidate group IDs.')
cand.to_parquet(TABLES/'02_candidate_group_manifest.parquet',index=False)
pixels.to_parquet(TABLES/'02_candidate_group_pixels.parquet',index=False)
payload={'manifest':sha(TABLES/'02_candidate_group_manifest.parquet'),'pixels':sha(TABLES/'02_candidate_group_pixels.parquet'),'source':sha(pred_path),'constants':PRE}
(OUT/'candidate_lock.json').write_text(json.dumps({'lock_id':oh(payload),'payload':payload},indent=2))
print('Pages:',len(ctx),'Groups:',len(cand),'Pixels:',len(pixels))


## 4. Construct post-lock safety and counterfactual value labels


In [ ]:
if not (OUT/'candidate_lock.json').exists(): raise RuntimeError('Candidate lock missing.')
clean_map={p.stem:p for p in (ROOT/'source'/'clean').iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}}
def ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def f1(ref,pred,region,tol=1):
 ref=ref&region; pred=pred&region
 if not ref.any(): return np.nan
 k=np.ones((2*tol+1,2*tol+1),np.uint8)
 p=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1)
 r=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum()
 return float(2*p*r/max(p+r,1e-12))
labels=[]
for page,page_candidates in cand.groupby('page'):
 clean=cv2.imread(str(clean_map[page]),0)
 if clean is None or clean.shape!=ctx[page]['obs'].shape: raise RuntimeError(f'{page}: clean truth unavailable or mismatched')
 d=ctx[page]; ci=ink(clean); ce=cv2.Canny(clean,70,170)>0; bi=ink(d['baseline']); be=cv2.Canny(d['baseline'],70,170)>0
 lookup=full[full.page==page].set_index(['y','x'])
 for _,r in page_candidates.iterrows():
  gm=d['candidate_labels']==int(r.candidate_group_id); yy,xx=np.where(gm)
  truth=np.array([int(lookup.loc[(int(y),int(x)),'truth']) for y,x in zip(yy,xx)])
  tu=int((truth==2).sum()); tp=int((truth==1).sum()); fa=int((truth==0).sum())
  safe_eligible=bool(tu==0); safe=int(safe_eligible and fa==0 and tp>0)
  new=d['baseline'].copy(); new[gm]=d['obs'][gm]
  region=cv2.dilate(gm.astype(np.uint8),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(25,25)))>0
  base_mae=float(np.abs(clean[region].astype(float)-d['baseline'][region].astype(float)).mean())
  new_mae=float(np.abs(clean[region].astype(float)-new[region].astype(float)).mean())
  mae_gain=base_mae-new_mae
  base_if1=f1(ci,bi,region); new_if1=f1(ci,ink(new),region)
  base_ef1=f1(ce,be,region); new_ef1=f1(ce,cv2.Canny(new,70,170)>0,region)
  ink_gain=float((new_if1-base_if1) if np.isfinite(base_if1) and np.isfinite(new_if1) else 0.)
  edge_gain=float((new_ef1-base_ef1) if np.isfinite(base_ef1) and np.isfinite(new_ef1) else 0.)
  value=float(mae_gain+8*ink_gain+8*edge_gain)
  labels.append({'group_id':r.group_id,'truth_print_pixels':tp,'truth_annotation_pixels':fa,'truth_uncertain_pixels':tu,
   'safety_label_eligible':safe_eligible,'safety_label':safe,'mae_benefit':mae_gain,'ink_f1_gain':ink_gain,
   'edge_f1_gain':edge_gain,'value_score':value,'value_positive':int(value>0)})
data=cand.merge(pd.DataFrame(labels),on='group_id',validate='one_to_one')
data.to_parquet(TABLES/'05_counterfactual_value_labels.parquet',index=False)
print(data[['safety_label','value_positive','value_score']].describe())


## 5. Three-scale tensor construction and self-validation


In [ ]:
PAD=OUTER//2
padded={}
for page,d in ctx.items():
 padded[page]={}
 for k in CHANNELS:
  a=d[k].astype(np.float32)
  if k in ['obs','norm']: a/=255.0
  elif k=='depth': a=np.clip(a/25.0,0,1)
  padded[page][k]=np.pad(a,PAD,mode='reflect')

def patch(page,cy,cx,size):
 cy=int(round(cy))+PAD; cx=int(round(cx))+PAD; h=size//2
 result=np.stack([padded[page][k][cy-h:cy+h+1,cx-h:cx+h+1] for k in CHANNELS]).astype(np.float32)
 if result.shape!=(len(CHANNELS),size,size): raise RuntimeError(f'{page}: patch shape {result.shape} != {(len(CHANNELS),size,size)}')
 return result

def outer_patch(page,cy,cx):
 raw=patch(page,cy,cx,OUTER)
 resized=np.stack([cv2.resize(ch,(OUTER_MODEL_SIZE,OUTER_MODEL_SIZE),interpolation=cv2.INTER_AREA) for ch in raw]).astype(np.float32)
 if resized.shape!=(len(CHANNELS),OUTER_MODEL_SIZE,OUTER_MODEL_SIZE): raise RuntimeError('Outer-context resize failure.')
 return resized

sample=data.iloc[0]
assert patch(sample.page,sample.centre_y,sample.centre_x,LOCAL).shape==(len(CHANNELS),33,33)
assert patch(sample.page,sample.centre_y,sample.centre_x,MEDIUM).shape==(len(CHANNELS),97,97)
assert outer_patch(sample.page,sample.centre_y,sample.centre_x).shape==(len(CHANNELS),97,97)
print('TENSOR CHECK PASSED:',(len(CHANNELS),33,33),(len(CHANNELS),97,97),(len(CHANNELS),97,97))


## 6. Matched dataset and paired model architecture


In [ ]:
class GroupDataset(Dataset):
 def __init__(self,idx): self.idx=np.asarray(idx,int)
 def __len__(self): return len(self.idx)
 def __getitem__(self,i):
  j=int(self.idx[i]); r=data.iloc[j]
  return (torch.from_numpy(patch(r.page,r.centre_y,r.centre_x,LOCAL)),
          torch.from_numpy(patch(r.page,r.centre_y,r.centre_x,MEDIUM)),
          torch.from_numpy(outer_patch(r.page,r.centre_y,r.centre_x)),
          torch.tensor(float(r.safety_label)),torch.tensor(float(r.value_positive)),
          torch.tensor(float(r.value_score)),torch.tensor(j))
def loader(idx,shuffle,seed):
 g=torch.Generator();g.manual_seed(seed)
 return DataLoader(GroupDataset(idx),batch_size=BATCH,shuffle=shuffle,num_workers=2,pin_memory=True,generator=g,persistent_workers=False)

class Encoder(nn.Module):
 def __init__(self):
  super().__init__();self.net=nn.Sequential(nn.Conv2d(len(CHANNELS),24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1))
 def forward(self,x): return self.net(x).flatten(1)

class OwnershipModel(nn.Module):
 def __init__(self,use_outer):
  super().__init__();self.use_outer=bool(use_outer);self.local=Encoder();self.medium=Encoder();self.outer=Encoder() if self.use_outer else None
  width=192 if self.use_outer else 128
  self.shared=nn.Sequential(nn.Linear(width,96),nn.ReLU(),nn.Dropout(.15),nn.Linear(96,64),nn.ReLU())
  self.safe=nn.Linear(64,1);self.value=nn.Linear(64,1);self.reg=nn.Linear(64,1)
 def forward(self,local,medium,outer):
  z=[self.local(local),self.medium(medium)]
  if self.use_outer: z.append(self.outer(outer))
  z=self.shared(torch.cat(z,1))
  return self.safe(z).squeeze(1),self.value(z).squeeze(1),self.reg(z).squeeze(1)

def reset_seed(s):
 random.seed(s);np.random.seed(s);torch.manual_seed(s);torch.cuda.manual_seed_all(s)
def fit_model(idx,use_outer,fold):
 reset_seed(SEED+fold);m=OwnershipModel(use_outer).to(DEVICE);opt=torch.optim.AdamW(m.parameters(),lr=LR,weight_decay=WD)
 y1=data.iloc[idx].safety_label.to_numpy();y2=data.iloc[idx].value_positive.to_numpy()
 loss_s=nn.BCEWithLogitsLoss(pos_weight=torch.tensor((len(y1)-y1.sum())/max(y1.sum(),1),device=DEVICE))
 loss_v=nn.BCEWithLogitsLoss(pos_weight=torch.tensor((len(y2)-y2.sum())/max(y2.sum(),1),device=DEVICE));loss_r=nn.SmoothL1Loss();hist=[]
 for ep in range(EPOCHS):
  m.train(); total=0.; n=0
  for local,medium,outer,ys,yv,vr,_ in loader(idx,True,SEED+ep):
   local,medium,outer=local.to(DEVICE),medium.to(DEVICE),outer.to(DEVICE);ys,yv,vr=ys.to(DEVICE),yv.to(DEVICE),vr.to(DEVICE)
   opt.zero_grad(set_to_none=True);ps,pv,pr=m(local,medium,outer);loss=loss_s(ps,ys)+loss_v(pv,yv)+.2*loss_r(pr,vr);loss.backward();opt.step();total+=float(loss.detach())*len(ys);n+=len(ys)
  hist.append(total/n)
 return m,hist
def predict_model(m,idx):
 result=[];m.eval()
 with torch.no_grad():
  for local,medium,outer,_,_,_,j in loader(idx,False,SEED):
   ps,pv,pr=m(local.to(DEVICE),medium.to(DEVICE),outer.to(DEVICE));result.extend((int(k),float(a),float(b),float(c)) for k,a,b,c in zip(j,torch.sigmoid(ps).cpu(),torch.sigmoid(pv).cpu(),pr.cpu()))
 return result

# Architecture assertion: both arms always consume local+medium; treatment alone uses outer.
assert OwnershipModel(False).shared[0].in_features==128
assert OwnershipModel(True).shared[0].in_features==192
print('MODEL CHECK PASSED: control=33+97; treatment=33+97+193')


## 7. Source-group OOF training and persisted paired models


In [ ]:
groups=data.source_group.astype(str)
splitter=LeaveOneGroupOut() if groups.nunique()<=10 else GroupKFold(n_splits=min(10,groups.nunique()))
folds=list(splitter.split(data,groups=groups))
fold_map=pd.DataFrame([{'fold':i,'test_groups':'|'.join(sorted(groups.iloc[te].unique())),'test_rows':len(te)} for i,(tr,te) in enumerate(folds,1)])
fold_map.to_csv(TABLES/'03_source_group_fold_map.csv',index=False)
oof_rows=[]; inventory=[]
for f,(tr,te) in enumerate(folds,1):
 if set(groups.iloc[tr])&set(groups.iloc[te]): raise RuntimeError(f'Fold {f}: source leakage')
 safety_train=np.array([i for i in tr if bool(data.iloc[i].safety_label_eligible)])
 if len(safety_train)<20 or data.iloc[safety_train].safety_label.nunique()<2: raise RuntimeError(f'Fold {f}: insufficient safety variation')
 for arm,use_outer in [('control_33_97',False),('treatment_33_97_193',True)]:
  print('FOLD',f,arm); model,hist=fit_model(tr,use_outer,f)
  path=MODELS/f'{arm}_fold{f}.pt';torch.save(model.state_dict(),path)
  inventory.append({'arm':arm,'fold':f,'path':str(path),'sha256':sha(path),'final_loss':hist[-1],'use_outer':use_outer})
  oof_rows.extend({'row_id':j,'fold':f,'arm':arm,'raw_safety':a,'raw_value':b,'pred_value':c} for j,a,b,c in predict_model(model,te))
  del model;py_gc.collect();torch.cuda.empty_cache()
oof_raw=pd.DataFrame(oof_rows)
if len(oof_raw)!=len(data)*2: raise RuntimeError(f'OOF row mismatch: {len(oof_raw)} != {len(data)*2}')
if set(oof_raw.arm)!=set(ARMS): raise RuntimeError('Unexpected OOF arm identities.')
pd.DataFrame(inventory).to_csv(AUDIT/'model_inventory.csv',index=False)


## 8. Cross-fitted calibration, frozen rule and concentration control


In [ ]:
calibrated=[];cal_inventory=[]
for arm,q0 in oof_raw.groupby('arm'):
 q=q0.sort_values('row_id').reset_index(drop=True); ps=np.full(len(q),np.nan);pv=np.full(len(q),np.nan)
 for f in sorted(q.fold.unique()):
  tr=q.fold!=f;te=q.fold==f
  safe=tr & data.loc[q.row_id,'safety_label_eligible'].to_numpy(bool)
  if safe.sum()<20 or data.loc[q.loc[safe,'row_id'],'safety_label'].nunique()<2: raise RuntimeError(f'{arm}/fold{f}: safety calibration variation insufficient')
  iso_s=IsotonicRegression(out_of_bounds='clip').fit(q.loc[safe,'raw_safety'],data.loc[q.loc[safe,'row_id'],'safety_label'])
  iso_v=IsotonicRegression(out_of_bounds='clip').fit(q.loc[tr,'raw_value'],data.loc[q.loc[tr,'row_id'],'value_positive'])
  ps[te]=iso_s.predict(q.loc[te,'raw_safety']);pv[te]=iso_v.predict(q.loc[te,'raw_value'])
  for kind,obj in [('safety',iso_s),('value',iso_v)]:
   path=CALIBRATORS/f'{arm}_fold{f}_{kind}.joblib';joblib.dump(obj,path);cal_inventory.append({'arm':arm,'fold':int(f),'kind':kind,'path':str(path),'sha256':sha(path)})
 q['p_safety']=ps;q['p_value']=pv;calibrated.append(q)
oof=pd.concat(calibrated,ignore_index=True).merge(data.reset_index().rename(columns={'index':'row_id'}),on='row_id',validate='many_to_one')
if not np.isfinite(oof[['p_safety','p_value','pred_value']].to_numpy()).all(): raise RuntimeError('Non-finite calibrated outputs.')
oof['eligible_select']=(oof.p_safety>=SAFETY_T)&(oof.p_value>=VALUE_T)&(oof.pred_value>0)&oof.safety_label_eligible
oof['selected']=False;oof['reason']='below_rule'
for arm,arm_rows in oof.groupby('arm'):
 for page,page_rows in arm_rows.groupby('page'):
  candidates=page_rows[page_rows.eligible_select].sort_values(['p_safety','p_value','pred_value'],ascending=False)
  page_cap=max(1,int(MAX_PAGE_FRACTION*len(page_rows)));chosen=[];counts={}
  for idx,r in candidates.iterrows():
   comp=int(r.mask_component_id);limit=max(1,int(MAX_COMPONENT_FRACTION*(page_rows.mask_component_id==comp).sum()))
   if len(chosen)>=page_cap:break
   if counts.get(comp,0)>=limit:continue
   chosen.append(idx);counts[comp]=counts.get(comp,0)+1
  oof.loc[chosen,['selected','reason']]=[True,'selected']
  oof.loc[candidates.index.difference(chosen),'reason']='concentration_cap'
oof.to_parquet(TABLES/'04_oof_safety_and_value_predictions.parquet',index=False)
pd.DataFrame(cal_inventory).to_csv(AUDIT/'calibrator_inventory.csv',index=False)


## 9. Evaluation, whole-page integrity and paired verdict


In [ ]:
def upper(errors,n,z=1.96):
 if n==0:return np.nan
 p=errors/n;d=1+z*z/n;c=p+z*z/(2*n);r=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n));return (c+r)/d
def safe_ap(q):
 m=q.safety_label_eligible.astype(bool)
 return float(average_precision_score(q.loc[m,'safety_label'],q.loc[m,'raw_safety'])) if m.sum() and q.loc[m,'safety_label'].nunique()==2 else np.nan
def value_ap(q):
 return float(average_precision_score(q.value_positive,q.raw_value)) if q.value_positive.nunique()==2 else np.nan
def summarise(q):
 selected=q.selected
 page=q.groupby('page').apply(lambda g:pd.Series({'selected':int(g.selected.sum()),'tp':int(g.loc[g.selected,'truth_print_pixels'].sum()),'fp':int(g.loc[g.selected,'truth_annotation_pixels'].sum()),'value':float(g.loc[g.selected,'value_score'].sum())}),include_groups=False).reset_index()
 rng=np.random.default_rng(SEED);idx=page.set_index('page');names=page.page.to_numpy();boot=[]
 for _ in range(BOOT): boot.append(float(idx.loc[rng.choice(names,len(names),replace=True)].value.sum()))
 errors=int((selected&(q.safety_label==0)).sum());n=int(selected.sum())
 return {'arm':q.arm.iloc[0],'groups':len(q),'safety_average_precision':safe_ap(q),'value_average_precision':value_ap(q),
  'selected_groups':n,'false_safe_groups':errors,'true_print_pixels':int(q.loc[selected,'truth_print_pixels'].sum()),
  'false_annotation_pixels':int(q.loc[selected,'truth_annotation_pixels'].sum()),'realised_value':float(q.loc[selected,'value_score'].sum()),
  'value_95_low':float(np.quantile(boot,.025)),'false_group_wilson95_upper':upper(errors,n),'selected_pages':int((page.selected>0).sum())}
summary=pd.DataFrame([summarise(q) for _,q in oof.groupby('arm')]);summary.to_csv(TABLES/'arm_comparison_summary.csv',index=False)
page_rows=[]
for arm,arm_rows in oof.groupby('arm'):
 for page,page_group in arm_rows.groupby('page'):
  d=ctx[page];selected_mask=np.zeros(d['mask'].shape,bool)
  for _,r in page_group[page_group.selected].iterrows(): selected_mask|=d['candidate_labels']==int(r.candidate_group_id)
  result=d['baseline'].copy();result[selected_mask]=d['obs'][selected_mask]
  outside=int((result[~d['mask']]!=d['obs'][~d['mask']]).sum());reg=int((result[~selected_mask]!=d['baseline'][~selected_mask]).sum());wrong=int(((result!=d['baseline'])&(result!=d['obs'])).sum())
  if outside or reg or wrong: raise RuntimeError(f'{arm}/{page}: page integrity failed')
  p=WHOLE/arm/page;p.mkdir(parents=True,exist_ok=False)
  cv2.imwrite(str(p/'observed.png'),d['obs']);cv2.imwrite(str(p/'v7_8.png'),d['baseline']);cv2.imwrite(str(p/'notebook43.png'),result)
  dm=np.zeros_like(result);dm[selected_mask]=255;cv2.imwrite(str(p/'selected_groups.png'),dm)
  page_rows.append({'arm':arm,'page':page,'source_group':page_group.source_group.iloc[0],'candidate_groups':len(page_group),'selected_groups':int(page_group.selected.sum()),'selected_pixels':int(selected_mask.sum()),'outside_mask_drift':outside,'nonselected_regression':reg,'wrong_changed_byte':wrong})
page_integrity=pd.DataFrame(page_rows);page_integrity.to_csv(AUDIT/'page_output_integrity.csv',index=False)
by_page=oof.groupby(['arm','page','source_group'],as_index=False).apply(lambda g:pd.Series({'candidate_groups':len(g),'selected_groups':int(g.selected.sum()),'false_safe_groups':int((g.selected&(g.safety_label==0)).sum()),'true_print_pixels':int(g.loc[g.selected,'truth_print_pixels'].sum()),'false_annotation_pixels':int(g.loc[g.selected,'truth_annotation_pixels'].sum()),'value_score':float(g.loc[g.selected,'value_score'].sum())}),include_groups=False)
by_page.to_csv(TABLES/'results_by_page.csv',index=False)
by_source=by_page.groupby(['arm','source_group'],as_index=False).sum(numeric_only=True);by_source.to_csv(TABLES/'results_by_source_group.csv',index=False)

s=summary.set_index('arm');c=s.loc['control_33_97'];t=s.loc['treatment_33_97_193'];src=by_source[by_source.arm=='treatment_33_97_193']
gates={
 'N43_1_design_identity':bool(set(summary.arm)==set(ARMS) and PRE['single_experimental_variable']=='addition of the 193x193 isotropic outer-context branch'),
 'N43_2_integrity':bool(page_integrity[['outside_mask_drift','nonselected_regression','wrong_changed_byte']].to_numpy().sum()==0),
 'N43_3_zero_false_safe':bool(t.false_safe_groups==0),
 'N43_4_false_bound':bool(np.isfinite(t.false_group_wilson95_upper) and t.false_group_wilson95_upper<=.05),
 'N43_5_positive_value':bool(t.realised_value>0 and t.value_95_low>0),
 'N43_6_treatment_value_dominance':bool(t.realised_value>c.realised_value),
 'N43_7_treatment_ranking_dominance':bool(t.safety_average_precision>=c.safety_average_precision and t.value_average_precision>c.value_average_precision),
 'N43_8_nontrivial_coverage':bool(t.selected_groups>=30 and t.selected_pages>=10 and src.loc[src.selected_groups>0,'source_group'].nunique()>=3),
 'N43_9_source_stability':bool((src.false_safe_groups==0).all() and (src.value_score>=0).all()),
}
first=next((k for k,v in gates.items() if v is False),None)
verdict={'status':'PASS_DEVELOPMENT' if first is None else 'FAIL','first_failed_gate':first,'gates':gates,'development_only':True,'control':'control_33_97','treatment':'treatment_33_97_193'}
(OUT/'notebook43_verdict.json').write_text(json.dumps(verdict,indent=2))
print(summary.to_string(index=False));print(json.dumps(verdict,indent=2))


## 10. Self-validation, manifest and final upload package


In [ ]:
# Machine-readable checks that make it difficult to mistake this file for the superseded design.
self_validation={
 'arms_exact':ARMS==['control_33_97','treatment_33_97_193'],
 'control_has_local':LOCAL==33,'control_has_medium':MEDIUM==97,
 'treatment_has_local':LOCAL==33,'treatment_has_medium':MEDIUM==97,'treatment_has_outer':OUTER==193,
 'outer_only_added_to_treatment':True,
 'p_print_present':all('p_print' in d for d in ctx.values()),
 'oof_rows_exact':len(oof)==len(data)*2,
 'source_group_leakage_zero':True,
 'page_integrity_zero':bool(page_integrity[['outside_mask_drift','nonselected_regression','wrong_changed_byte']].to_numpy().sum()==0),
}
if not all(self_validation.values()): raise RuntimeError(f'SELF-VALIDATION FAILED: {self_validation}')
(OUT/'SELF_VALIDATION_PASSED.json').write_text(json.dumps(self_validation,indent=2))

env={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':PRE['version'],'pre_registration':PRE,'source_record':record,'candidate_lock':json.loads((OUT/'candidate_lock.json').read_text()),'groups':len(data),'pages':int(data.page.nunique()),'source_groups':int(data.source_group.nunique()),'models':inventory,'calibrators':cal_inventory,'self_validation':self_validation,'verdict':verdict,'environment':env}
(OUT/'experiment_manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
lines=['NOTEBOOK 43 REBUILT — 33+97 VS 33+97+193','',f'Run: {OUT}','CONTROL: 33x33 + 97x97','TREATMENT: 33x33 + 97x97 + 193x193','ONLY VARIABLE: addition of 193x193 isotropic outer context','',f'Candidate groups: {len(data)}',f'Pages: {data.page.nunique()} | source groups: {data.source_group.nunique()}',f'Verdict: {verdict["status"]}',f'First failed gate: {verdict["first_failed_gate"]}','',summary.to_string(index=False),'','Development evidence only. No threshold relaxation or production authority.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines))
files=[OUT/'01_pre_registration.json',OUT/'source_record.json',OUT/'candidate_lock.json',OUT/'SELF_VALIDATION_PASSED.json',OUT/'experiment_manifest.json',OUT/'notebook43_verdict.json',OUT/'executive_summary.txt',TABLES/'02_candidate_group_manifest.parquet',TABLES/'05_counterfactual_value_labels.parquet',TABLES/'03_source_group_fold_map.csv',TABLES/'04_oof_safety_and_value_predictions.parquet',TABLES/'arm_comparison_summary.csv',TABLES/'results_by_page.csv',TABLES/'results_by_source_group.csv',AUDIT/'model_inventory.csv',AUDIT/'calibrator_inventory.csv',AUDIT/'page_output_integrity.csv']
missing=[str(p) for p in files if not p.exists()]
if missing: raise RuntimeError(f'Upload package missing {missing}')
inv=[]
for p in files:
 rel=p.relative_to(OUT);dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst);inv.append({'relative_path':str(rel),'bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(inv).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 43 REBUILT. Verify SELF_VALIDATION_PASSED.json and arm_comparison_summary.csv first. CONTROL=33+97; TREATMENT=33+97+193.')
(UPLOAD/'upload_file_list.txt').write_text(chr(10).join(x['relative_path'] for x in inv))
zip_path=UPLOAD/'notebook43_REBUILT_33_97_vs_33_97_193_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in UPLOAD.rglob('*'):
  if p.is_file() and p!=zip_path:z.write(p,p.relative_to(UPLOAD))
print(chr(10).join(lines));print('SELF-VALIDATION PASSED');print('Upload:',zip_path)


## Review order

1. `SELF_VALIDATION_PASSED.json`
2. `01_pre_registration.json`
3. `tables/arm_comparison_summary.csv`
4. `notebook43_verdict.json`
5. `tables/results_by_source_group.csv`
6. paired whole pages under `whole_pages/control_33_97/` and `whole_pages/treatment_33_97_193/`
7. final upload bundle

If the pre-registration or self-validation does not say exactly **33+97 versus 33+97+193**, do not run the training cells.
